# Molecules, dynamics and amplitude estimation

QSO 1.4 adds algorithms from chemistry and physics. This tutorial runs each
of them on small examples and compares the results with exact references:

- fermion-to-qubit mappings and molecular Hamiltonians from FCIDUMP files,
  against full CI computed by PySCF;
- UCCSD and ADAPT-VQE;
- excited states by variational quantum deflation;
- product formulas and their error bounds;
- variational imaginary-time evolution;
- QAOA with a constraint-preserving mixer;
- amplitude estimation and its 1/N error scaling.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import metatron_qso as q

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metatron-qso-rs").is_dir())
MOLECULES = root / "metatron-qso-rs" / "examples" / "molecules"
reference = json.loads((MOLECULES / "reference.json").read_text())
print("QSO", q.__version__)

## LiH on qubits

The FCIDUMP file holds the one- and two-electron integrals of LiH at 1.595 Å
in the STO-3G basis, computed by PySCF. The four encodings below give
different Pauli strings. Within the determinants with two spin-up and two
spin-down electrons, all four have the same spectrum, and its lowest level
is PySCF's full-CI energy.

In [ ]:
lih = q.MolecularIntegrals.from_file(MOLECULES / "lih_sto3g.fcidump")
for mapping, reduce in [("jordan_wigner", False), ("parity", False),
                        ("bravyi_kitaev", False), ("parity", True)]:
    h = lih.qubit_hamiltonian(mapping, two_qubit_reduction=reduce)
    print(f"{mapping:14s} reduced={reduce!s:5s} qubits={h.min_qubits:2d} terms={h.num_terms}")
fci = lih.fci_energies(3)
print("Hartree-Fock", lih.hartree_fock_energy(), " PySCF", reference["lih_sto3g"]["hartree_fock"])
print("full CI     ", fci[0], " PySCF", reference["lih_sto3g"]["fci"][0])

## UCCSD and ADAPT-VQE

UCCSD has one parameter per single and double excitation: 92 for LiH. ADAPT-VQE
grows its circuit one operator at a time. At each step it adds the operator
with the largest energy gradient. The plot shows its energy after each
step, measured from full CI, against the chemical-accuracy line at 1.6 mHa.

In [ ]:
h = lih.qubit_hamiltonian("parity", two_qubit_reduction=True)
ansatz = q.uccsd(lih, "parity", two_qubit_reduction=True)
uccsd = q.run_vqe(h, ansatz, initial_parameters=np.zeros(ansatz.num_parameters))
adapt = q.adapt_vqe(lih, "parity", two_qubit_reduction=True)
print(f"UCCSD     {ansatz.num_parameters} parameters, error {uccsd['energy'] - fci[0]:.2e} Ha")
print(f"ADAPT-VQE {len(adapt['parameters'])} parameters, error {adapt['energy'] - fci[0]:.2e} Ha")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(np.arange(1, len(adapt["energies"]) + 1), adapt["energies"] - fci[0], "o-", label="ADAPT-VQE")
ax.axhline(uccsd["energy"] - fci[0], color="C1", ls="--", label="UCCSD (92 parameters)")
ax.axhline(1.6e-3, color="gray", ls=":", label="chemical accuracy")
ax.set_xlabel("operators")
ax.set_ylabel("E − E_FCI (Ha)")
ax.legend()
plt.show()

## Excited states of H₂

With the parity mapping and the two-qubit reduction, H₂ lives on two
qubits. VQD finds all four levels in the Sz = 0 sector one after another.
Each state is penalised for its overlap with the states found before it.

In [ ]:
h2 = q.MolecularIntegrals.from_file(MOLECULES / "h2_sto3g.fcidump")
h = h2.qubit_hamiltonian("parity", two_qubit_reduction=True)
levels = q.vqd(h, q.efficient_su2(2, 2), 4, restarts=6, seed=2)
for k, (e, f) in enumerate(zip(levels["energies"], h2.fci_energies(4))):
    print(f"E{k}: VQD {e:+.10f}   full CI {f:+.10f}")

## Product formulas and their error bounds

For a four-qubit Heisenberg chain, the plot compares the actual error of
`exp(−iHt)` from product formulas of order 1, 2 and 4 with QSO's
rigorous bounds. The errors fall as `r^{-order}` with the number of steps
`r`. The bounds for orders 1 and 2 are commutator bounds, about ten times
the actual error. The bound for order 4 is valid but loose.

In [ ]:
ham = q.heisenberg_xxz(4, 1.0, 0.5)
w, v = np.linalg.eigh(ham.to_matrix(4))
exact = (v * np.exp(-1j * w)) @ v.conj().T

def unitary(c):
    cols = []
    for k in range(16):
        prep = q.Circuit(4)
        for b in range(4):
            if k >> b & 1:
                prep.x(b)
        cols.append(q.simulate(prep.append(c)).amplitudes)
    return np.array(cols).T

steps = [2, 4, 8, 16, 32]
fig, ax = plt.subplots(figsize=(6, 3.5))
for order, color in [(1, "C0"), (2, "C1"), (4, "C2")]:
    errors = [np.linalg.norm(unitary(q.trotter_circuit(ham, 4, 1.0, r, order)) - exact, 2) for r in steps]
    bounds = [q.trotter_error_bound(ham, 1.0, r, order) for r in steps]
    ax.loglog(steps, errors, "o-", color=color, label=f"order {order}")
    ax.loglog(steps, bounds, "--", color=color)
ax.set_xlabel("steps r")
ax.set_ylabel("‖U − exp(−iHt)‖ (solid), bound (dashed)")
ax.legend()
plt.show()

## Imaginary-time evolution

McLachlan's variational principle turns imaginary-time evolution
`e^{−Hτ}|ψ₀⟩` into a flow of the circuit parameters. With real amplitudes
on two qubits, the ansatz can represent every real state. The variational
energy therefore follows the exact normalised evolution and decays to the
ground energy.

In [ ]:
ham = q.PauliSum("Z0 + 0.6 X0 X1 - 0.4 Z1 + 0.3 X1")
ansatz = q.real_amplitudes(2, 2)
theta0 = np.array([0.3, -0.5, 0.2, 0.4, 0.1, -0.3])
ite = q.imaginary_time_evolution(ansatz, ham, theta0, time=4.0, steps=200)
psi0 = q.simulate(ansatz, theta0).amplitudes
w, v = np.linalg.eigh(ham.to_matrix(2))
def exact_energy(tau):
    phi = v @ (np.exp(-w * tau) * (v.conj().T @ psi0))
    return float(np.real(phi.conj() @ (ham.to_matrix(2) @ phi)) / np.real(phi.conj() @ phi))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ite["times"], ite["energies"], label="variational")
ax.plot(ite["times"][::10], [exact_energy(t) for t in ite["times"][::10]], "o", label="exact")
ax.axhline(w[0], color="gray", ls=":", label="ground energy")
ax.set_xlabel("imaginary time τ")
ax.set_ylabel("⟨H⟩")
ax.legend()
plt.show()

## QAOA with a constraint

The problem: choose exactly three of eight items so that a random QUBO is
minimal. The ring XY mixer starts in the Dicke state of Hamming weight 3
and conserves the weight, so all measured strings are feasible. The X mixer
searches all 256 strings. Its constrained optimum is the same, but most of
its probability lies on infeasible strings.

In [ ]:
rng = np.random.default_rng(5)
qubo = q.Qubo(rng.uniform(-1, 1, (8, 8)))
cost = qubo.to_pauli_sum()
for p in (1, 2, 3):
    xy = q.Qaoa(cost, 8, p, mixer="xy_ring", weight=3).optimize(restarts=4, seed=1)
    print(f"p={p}: approximation ratio {xy['approximation_ratio']:.3f}, "
          f"P(optimal) {xy['optimal_probability']:.3f}, P(feasible) {xy['feasible_probability']:.3f}")
qx = q.Qaoa(cost, 8, 2, mixer="x")
x = qx.optimize(restarts=4, seed=1)
prep = q.Circuit(8)
for b in range(8):
    prep.h(b)
probs = np.abs(q.simulate(prep.append(qx.circuit()), np.concatenate([x["gammas"], x["betas"]])).amplitudes) ** 2
weight3 = sum(p for b, p in enumerate(probs) if bin(b).count("1") == 3)
print(f"X mixer, p=2: P(weight-3 string) = {weight3:.3f}")

## Amplitude estimation

Estimate `a = 0.3`, the probability of measuring 1 after `RY(2 arcsin √0.3)`.
Maximum-likelihood amplitude estimation measures `Q^k A|0⟩` for
`k = 0, 1, 2, 4, …`. Its root-mean-square error over 200 seeds falls as
`1/N` in the number of oracle calls `N`. Plain sampling (`k = 0` only)
falls as `1/√N`.

In [ ]:
problem = q.AmplitudeProblem.bernoulli(0.3)

def rmse(levels, shots, seeds=200):
    runs = [q.ml_amplitude_estimation(problem, levels=levels, shots=shots, seed=s) for s in range(seeds)]
    return runs[0]["oracle_calls"], np.sqrt(np.mean([(r["estimate"] - 0.3) ** 2 for r in runs]))

quantum = [rmse(levels, 100) for levels in range(3, 9)]
classical = [rmse(1, shots) for shots in (100, 400, 1600, 6400, 25600)]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(*zip(*quantum), "o-", label="maximum likelihood, k = 0, 1, 2, 4, …")
ax.loglog(*zip(*classical), "s-", label="sampling, k = 0")
ax.set_xlabel("oracle calls N")
ax.set_ylabel("RMSE")
ax.legend()
plt.show()
slope = np.polyfit(np.log([n for n, _ in quantum]), np.log([e for _, e in quantum]), 1)[0]
print(f"slope {slope:.2f}")